#MMD-Thinker


## 0. Зависимости

In [ ]:
%%capture
!pip install catboost lightgbm scikit-learn pandas numpy matplotlib seaborn pyarrow

## 1. Импорты и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report
)
SEED = 42
np.random.seed(SEED)
N_FOLDS   = 5
SVD_COMP  = 50
PCA_COMP  = 32

Импорты загружены


## 2. Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/ozon_train.csv', encoding='utf-8')
print(f'Данные: {df.shape}')
print(f'Доля контрафакта: {df["resolution"].mean():.4f}')
clip_emb = pd.read_parquet('/content/drive/MyDrive/clip_embeddings.parquet')
clip_emb['vector'] = clip_emb['embedding'].apply(lambda x: np.array(x, dtype=np.float32))
print(f'CLIP embeddings: {clip_emb.shape}')

Mounted at /content/drive
Данные: (197198, 45)
Доля контрафакта: 0.0662
CLIP embeddings: (187604, 3)


## 3. Seller-based разбиение

GroupShuffleSplit по продавцу — предотвращает entity-level leakage (из v3, оставляем).

In [ ]:
seller_class_counts = df.groupby('SellerID')['resolution'].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index
np.random.seed(SEED)
test_sellers  = np.random.choice(multi_class_sellers,
size=int(0.3 * len(multi_class_sellers)),
replace=False)
train_sellers = np.setdiff1d(df['SellerID'].unique(), test_sellers)
train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
test_df  = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)
print(f'Train: {train_df.shape}, контрафакт: {train_df["resolution"].mean():.4f}')
print(f'Test:  {test_df.shape},  контрафакт: {test_df["resolution"].mean():.4f}')

Train: (177380, 45), контрафакт: 0.0588
Test:  (19818, 45),  контрафакт: 0.1321


## 4. Feature Engineering

### 4.1 LOO Target Encoding — Seller, Brand, Name, Category

**Новое в v4:** добавляем LOO для brand, name_rus, CommercialTypeName4.  
EDA показал: brand_Sony=19.7%, name='Power Bank 30000'=100% контрафакта.

In [ ]:
global_mean = train_df['resolution'].mean()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def loo_encode_train(df_, col, target='resolution', n_splits=N_FOLDS):
    loo = np.full(len(df_), global_mean)
    for _, (tr_idx, val_idx) in enumerate(KFold(n_splits, shuffle=True, random_state=SEED).split(df_)):
        fold_rate = df_.iloc[tr_idx].groupby(col)[target].mean()
        loo[val_idx] = df_.iloc[val_idx][col].map(fold_rate).fillna(global_mean).values
    return loo

def loo_encode_test(train_df_, test_df_, col, target='resolution'):
    global_rates = train_df_.groupby(col)[target].mean()
    return test_df_[col].map(global_rates).fillna(global_mean).values
train_df['seller_problem_rate'] = loo_encode_train(train_df, 'SellerID')
test_df['seller_problem_rate']  = loo_encode_test(train_df, test_df, 'SellerID')
print(f'seller_problem_rate corr: {train_df["seller_problem_rate"].corr(train_df["resolution"]):.4f}')
train_df['brand_problem_rate'] = loo_encode_train(train_df, 'brand_name')
test_df['brand_problem_rate']  = loo_encode_test(train_df, test_df, 'brand_name')
print(f'brand_problem_rate corr: {train_df["brand_problem_rate"].corr(train_df["resolution"]):.4f}')
train_df['category_problem_rate'] = loo_encode_train(train_df, 'CommercialTypeName4')
test_df['category_problem_rate']  = loo_encode_test(train_df, test_df, 'CommercialTypeName4')
print(f'category_problem_rate corr: {train_df["category_problem_rate"].corr(train_df["resolution"]):.4f}')
train_df['name_problem_rate'] = loo_encode_train(train_df, 'name_rus')
test_df['name_problem_rate']  = loo_encode_test(train_df, test_df, 'name_rus')
print(f'name_problem_rate corr: {train_df["name_problem_rate"].corr(train_df["resolution"]):.4f}')

seller_problem_rate corr: 0.7107
brand_problem_rate corr: 0.4216
category_problem_rate corr: 0.5630
name_problem_rate corr: 0.5071


### 4.2 Табличный feature engineering

Из v3 + новые interaction features.

In [ ]:
SUSPICIOUS_WORDS = ['реплика', 'копия', 'аналог', 'совместим', 'noname',
                    'compatible', 'оригинал', 'original', 'brand new']
def engineer_features(df_):
    out = df_.copy()
    nan_flag_cols = ['brand_name', 'description', 'GmvTotal7', 'GmvTotal30',
                     'GmvTotal90', 'rating_5_count', 'ExemplarAcceptedCountTotal7']
    for col in nan_flag_cols:
        if col in out.columns:
            out[f'is_null_{col}'] = out[col].isnull().astype(np.float32)
    for w in [7, 30, 90]:
        s = out.get(f'item_count_sales{w}',        pd.Series(0, index=out.index))
        r = out.get(f'item_count_returns{w}',      pd.Series(0, index=out.index))
        f = out.get(f'item_count_fake_returns{w}', pd.Series(0, index=out.index))
        out[f'return_rate{w}']         = r / (s + 1)
        out[f'fake_return_rate{w}']    = f / (s + 1)
        out[f'fake_to_all_returns{w}'] = f / (r + 1)
        out[f'zero_sales{w}']          = (s == 0).astype(np.float32)
    for col in ['item_time_alive', 'seller_time_alive', 'PriceDiscounted']:
        if col in out.columns:
            out[f'log_{col}'] = np.log1p(out[col].fillna(0))
    if 'log_item_time_alive' in out.columns and 'log_seller_time_alive' in out.columns:
        out['log_age_interaction'] = out['log_item_time_alive'] * out['log_seller_time_alive']
    out['description_length'] = out['description'].fillna('').str.len()
    out['name_length']        = out['name_rus'].fillna('').str.len()
    out['brand_length']       = out['brand_name'].fillna('').str.len()
    out['has_description']    = out['description'].notna().astype(np.float32)
    out['has_brand']          = out['brand_name'].notna().astype(np.float32)
    out['text_complexity']    = out['description_length'] / (out['name_length'] + 1)
    out['desc_name_ratio']    = np.log1p(out['description_length']) / np.log1p(out['name_length'] + 1)
    brand_filled = out['brand_name'].fillna('').str.lower()
    name_filled  = out['name_rus'].fillna('').str.lower()
    out['brand_in_name'] = [
        1 if b and b in n else 0
        for b, n in zip(brand_filled, name_filled)
    ]
    out['brand_name_conf'] = [
        1 if b and b not in n else 0
        for b, n in zip(brand_filled, name_filled)
    ]
    out['has_suspicious_word'] = name_filled.apply(
        lambda x: 1 if any(w in x for w in SUSPICIOUS_WORDS) else 0
    )
    item_age = out.get('item_time_alive', pd.Series(999, index=out.index))
    seller_age = out.get('seller_time_alive', pd.Series(999, index=out.index))
    out['new_item_new_seller'] = (
        (item_age < 30) & (seller_age < 180)
    ).astype(np.float32)
    out['item_seller_age_ratio'] = (
        item_age / (seller_age + 1)
    ).fillna(0)
    price = out.get('PriceDiscounted', pd.Series(0, index=out.index)).fillna(0)
    out['price_log'] = np.log1p(price)
    out['price_zero'] = (price == 0).astype(np.float32)
    return out
train_df = engineer_features(train_df)
test_df  = engineer_features(test_df)

new_feats = [c for c in train_df.columns if any(
    p in c for p in ['is_null', 'rate', 'log_', 'seller_problem', 'brand_problem',
                     'category_problem', 'name_problem', 'interaction', 'length',
                     'has_', 'complexity', 'brand_in', 'suspicious', 'new_item',
                     'age_ratio', 'price', 'zero_sales'])]
print(f'Признаков после инжиниринга: {train_df.shape[1]}')
print(f'Новых признаков: {len(new_feats)}')

Признаков после инжиниринга: 86
Новых признаков: 36


### 4.3 Текстовые признаки — TF-IDF SVD (Mode 2)

TF-IDF + SVD работает лучше сырых BERT в бустинге: меньше размерность, меньше шума.

In [ ]:
def make_text(df_):
    brand = df_['brand_name'].fillna('без бренда')
    return (df_['name_rus'].fillna('') + ' ' +
            df_['description'].fillna('') + ' ' + brand)

train_texts = make_text(train_df).values
test_texts  = make_text(test_df).values
tfidf = TfidfVectorizer(
    max_features=80_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    analyzer='word',
)
X_tfidf_train = tfidf.fit_transform(train_texts)
X_tfidf_test  = tfidf.transform(test_texts)
print(f'TF-IDF: {X_tfidf_train.shape}')
svd = TruncatedSVD(n_components=SVD_COMP, random_state=SEED)
X_svd_train = svd.fit_transform(X_tfidf_train).astype(np.float32)
X_svd_test  = svd.transform(X_tfidf_test).astype(np.float32)
print(f'SVD объяснённая дисперсия: {svd.explained_variance_ratio_.sum():.3f}')

svd_cols = [f'svd_{i}' for i in range(SVD_COMP)]
train_df[svd_cols] = X_svd_train
test_df[svd_cols]  = X_svd_test
print(f'Добавлено {SVD_COMP} SVD признаков')

Обучаем TF-IDF...
TF-IDF: (177380, 80000)
SVD...
SVD объяснённая дисперсия: 0.156
Добавлено 50 SVD признаков


### 4.4 CLIP признаки — PCA (Mode 3)

CLIP 512d → PCA 32d. Добавляем как признаки к бустингу, не как отдельную голову.

In [ ]:
train_clip = train_df[['ItemID']].merge(clip_emb[['ItemID', 'vector']], on='ItemID', how='left')
test_clip  = test_df[['ItemID']].merge(clip_emb[['ItemID', 'vector']], on='ItemID', how='left')
has_clip_train = train_clip['vector'].notna()
has_clip_test  = test_clip['vector'].notna()
X_clip_tr_raw = np.vstack(train_clip[has_clip_train]['vector'].values)
pca = PCA(n_components=PCA_COMP, random_state=SEED)
X_clip_tr_pca = pca.fit_transform(X_clip_tr_raw).astype(np.float32)
print(f'CLIP PCA объяснённая дисперсия: {pca.explained_variance_ratio_.sum():.3f}')
clip_cols = [f'clip_{i}' for i in range(PCA_COMP)]
train_df[clip_cols] = 0.0
test_df[clip_cols]  = 0.0
train_df.loc[has_clip_train.values, clip_cols] = X_clip_tr_pca
X_clip_te_raw = np.vstack(test_clip[has_clip_test]['vector'].values)
X_clip_te_pca = pca.transform(X_clip_te_raw).astype(np.float32)
test_df.loc[has_clip_test.values, clip_cols] = X_clip_te_pca
train_df['has_clip'] = has_clip_train.astype(np.float32).values
test_df['has_clip']  = has_clip_test.astype(np.float32).values
print(f'CLIP покрытие train: {has_clip_train.mean():.3f}')
print(f'CLIP покрытие test:  {has_clip_test.mean():.3f}')

CLIP PCA объяснённая дисперсия: 0.591
CLIP покрытие train: 0.951
CLIP покрытие test:  0.958


### 4.5 Финальные наборы признаков для каждого режима

In [ ]:
DROP_ALWAYS = {'id', 'ItemID', 'SellerID', 'name_rus', 'description',
               'brand_name', 'CommercialTypeName4', 'resolution'}
def get_num_cols(df_, exclude_prefixes=()):
    cols = [
        c for c in df_.columns
        if c not in DROP_ALWAYS
        and df_[c].dtype in ['float64', 'int64', 'float32']
        and not any(c.startswith(p) for p in exclude_prefixes)
    ]
    return cols
cols_mode1 = get_num_cols(train_df, exclude_prefixes=('svd_', 'clip_'))
cols_mode2 = get_num_cols(train_df, exclude_prefixes=('clip_',))
cols_mode3 = get_num_cols(train_df)
print(f'Mode 1 (табличный):       {len(cols_mode1)} признаков')
print(f'Mode 2 (+ текст SVD):     {len(cols_mode2)} признаков')
print(f'Mode 3 (+ текст + CLIP):  {len(cols_mode3)} признаков')
y_train = train_df['resolution'].values
y_test  = test_df['resolution'].values
X_tr = {1: train_df[cols_mode1].fillna(-1).values,
         2: train_df[cols_mode2].fillna(-1).values,
         3: train_df[cols_mode3].fillna(-1).values}
X_te = {1: test_df[cols_mode1].fillna(-1).values,
         2: test_df[cols_mode2].fillna(-1).values,
         3: test_df[cols_mode3].fillna(-1).values}

Mode 1 (табличный):       79 признаков
Mode 2 (+ текст SVD):     129 признаков
Mode 3 (+ текст + CLIP):  161 признаков


## 5. Конфигурация CatBoost

Один конфиг для всех трёх режимов — только входные данные разные.

In [ ]:
CB_PARAMS = dict(
    iterations          = 5000,
    learning_rate       = 0.03,
    depth               = 8,
    loss_function       = 'Logloss',
    eval_metric         = 'AUC',
    auto_class_weights  = 'SqrtBalanced',
    early_stopping_rounds = 200,
    random_seed         = SEED,
    verbose             = 0,
    l2_leaf_reg         = 5,
    bagging_temperature = 0.5,
    subsample           = 0.8,
    colsample_bylevel   = 0.8,
)
for k, v in CB_PARAMS.items():
    print(f'  {k}: {v}')

CatBoost параметры:
  iterations: 5000
  learning_rate: 0.03
  depth: 8
  loss_function: Logloss
  eval_metric: AUC
  auto_class_weights: SqrtBalanced
  early_stopping_rounds: 200
  random_seed: 42
  verbose: 0
  l2_leaf_reg: 5
  bagging_temperature: 0.5
  subsample: 0.8
  colsample_bylevel: 0.8


## 6. Обучение трёх режимов с OOF предсказаниями

OOF (Out-Of-Fold) предсказания — честные мета-признаки без утечки.  
Это аналог того, как в MMD-Thinker каждый режим даёт независимый reward.

In [ ]:
def train_mode_oof(mode_num, X_train, y_train, X_test, params, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_probs  = np.zeros(len(y_train))
    test_probs = np.zeros(len(X_test))
    fold_aucs  = []
    importances = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr_f, X_val_f = X_train[tr_idx], X_train[val_idx]
        y_tr_f, y_val_f = y_train[tr_idx], y_train[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            X_tr_f, y_tr_f,
            eval_set=(X_val_f, y_val_f),
        )

        val_pred = model.predict_proba(X_val_f)[:, 1]
        oof_probs[val_idx] = val_pred
        fold_auc = roc_auc_score(y_val_f, val_pred)
        fold_aucs.append(fold_auc)
        test_probs += model.predict_proba(X_test)[:, 1] / n_folds
        importances.append(model.get_feature_importance())
        print(f'  Mode {mode_num} | Fold {fold+1} | Val AUC: {fold_auc:.4f} | Iter: {model.best_iteration_}')
    oof_auc = roc_auc_score(y_train, oof_probs)
    oof_pr  = average_precision_score(y_train, oof_probs)
    print(f'Mode {mode_num} OOF → ROC-AUC: {oof_auc:.4f} | PR-AUC: {oof_pr:.4f}')
    return oof_probs, test_probs, fold_aucs, np.mean(importances, axis=0)
print('Mode 1: Только табличные признаки (Quick Response)')
oof1, test1, aucs1, imp1 = train_mode_oof(1, X_tr[1], y_train, X_te[1], CB_PARAMS)
print('Mode 2: Таблица + TF-IDF SVD (Semantic Analysis)')
oof2, test2, aucs2, imp2 = train_mode_oof(2, X_tr[2], y_train, X_te[2], CB_PARAMS)
print('Mode 3: Таблица + SVD + CLIP (Deep Deliberation)')
oof3, test3, aucs3, imp3 = train_mode_oof(3, X_tr[3], y_train, X_te[3], CB_PARAMS)

=== Mode 1: Только табличные признаки (Quick Response) ===
  Mode 1 | Fold 1 | Val AUC: 0.9866 | Iter: 1698
  Mode 1 | Fold 2 | Val AUC: 0.9884 | Iter: 2169
  Mode 1 | Fold 3 | Val AUC: 0.9886 | Iter: 2281
  Mode 1 | Fold 4 | Val AUC: 0.9888 | Iter: 1473
  Mode 1 | Fold 5 | Val AUC: 0.9881 | Iter: 1980
Mode 1 OOF → ROC-AUC: 0.9881 | PR-AUC: 0.9050

=== Mode 2: Таблица + TF-IDF SVD (Semantic Analysis) ===
  Mode 2 | Fold 1 | Val AUC: 0.9883 | Iter: 2412
  Mode 2 | Fold 2 | Val AUC: 0.9900 | Iter: 2498
  Mode 2 | Fold 3 | Val AUC: 0.9901 | Iter: 3445
  Mode 2 | Fold 4 | Val AUC: 0.9907 | Iter: 1930
  Mode 2 | Fold 5 | Val AUC: 0.9894 | Iter: 2353
Mode 2 OOF → ROC-AUC: 0.9896 | PR-AUC: 0.9119

=== Mode 3: Таблица + SVD + CLIP (Deep Deliberation) ===
  Mode 3 | Fold 1 | Val AUC: 0.9884 | Iter: 1662
  Mode 3 | Fold 2 | Val AUC: 0.9900 | Iter: 3257
  Mode 3 | Fold 3 | Val AUC: 0.9904 | Iter: 3928
  Mode 3 | Fold 4 | Val AUC: 0.9909 | Iter: 3050
  Mode 3 | Fold 5 | Val AUC: 0.9896 | Iter: 292

## 7. Adaptive Thinking — мета-модель (Mixed Advantage)

LogisticRegression на OOF предсказаниях трёх режимов.  
Это точный аналог mixed advantage из MMD-Thinker §3.3:
- мета-модель выучивает, **кому доверять** для каждого примера
- эквивалент gate, который не коллапсирует (т.к. обучается на честных OOF)

Дополнительно: **sample difficulty weighting** — аналог A_sample (формула 4):  
примеры, где Mode 1 неуверен (prob ≈ 0.5), получают больший вес в мета-модели.

In [ ]:
meta_train = np.column_stack([
    oof1, oof2, oof3,
    np.abs(oof1 - oof2),
    np.abs(oof2 - oof3),
    np.abs(oof1 - oof3),
    oof1 * oof2,
    oof1 * oof3,
    oof2 * oof3,
])
meta_test = np.column_stack([
    test1, test2, test3,
    np.abs(test1 - test2),
    np.abs(test2 - test3),
    np.abs(test1 - test3),
    test1 * test2,
    test1 * test3,
    test2 * test3,
])
uncertainty_mode1 = 1 - 2 * np.abs(oof1 - 0.5)
sample_weights = 1 + uncertainty_mode1
scaler_meta = StandardScaler()
meta_train_sc = scaler_meta.fit_transform(meta_train)
meta_test_sc  = scaler_meta.transform(meta_test)
meta_model = LogisticRegression(
    C=0.5,
    max_iter=1000,
    random_state=SEED,
    class_weight='balanced'
)
meta_model.fit(meta_train_sc, y_train, sample_weight=sample_weights)
final_probs_train = meta_model.predict_proba(meta_train_sc)[:, 1]
final_probs_test  = meta_model.predict_proba(meta_test_sc)[:, 1]
print('Коэффициенты мета-модели (режимы):')
meta_names = ['Mode1', 'Mode2', 'Mode3', '|M1-M2|', '|M2-M3|', '|M1-M3|',
              'M1*M2', 'M1*M3', 'M2*M3']
for name, coef in zip(meta_names, meta_model.coef_[0]):
    print(f'  {name}: {coef:.4f}')

Коэффициенты мета-модели (режимы):
  Mode1: 1.8364
  Mode2: 0.9528
  Mode3: 0.9705
  |M1-M2|: -0.0029
  |M2-M3|: 0.0589
  |M1-M3|: -0.0539
  M1*M2: -1.6169
  M1*M3: -1.3987
  M2*M3: 1.0873


## 8. Итоговая оценка

In [ ]:
def evaluate_full(y_true, y_pred, label=''):
    roc = roc_auc_score(y_true, y_pred)
    pr  = average_precision_score(y_true, y_pred)
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_pred)
    precision_for_thresh = precision_arr[:-1]
    recall_for_thresh = recall_arr[:-1]
    mask = precision_for_thresh >= 0.9
    if mask.any():
        best_idx = np.argmax(recall_for_thresh[mask])
        recall_at_90 = recall_for_thresh[mask][best_idx]
        thresh_at_90 = thresholds[mask][best_idx]
    else:
        recall_at_90 = 0.0
        thresh_at_90 = None

    if label:
        print(f'{label}: ROC={roc:.4f} | PR={pr:.4f} | Recall@P≥0.9={recall_at_90:.4f}')
    return roc, pr, recall_at_90, precision_arr, recall_arr, thresholds
roc1, pr1, r1, _, _, _ = evaluate_full(y_test, test1, 'Mode 1 (табличный)')
roc2, pr2, r2, _, _, _ = evaluate_full(y_test, test2, 'Mode 2 (+текст SVD)')
roc3, pr3, r3, _, _, _ = evaluate_full(y_test, test3, 'Mode 3 (+текст+CLIP)')
roc_f, pr_f, r_f, prec_arr, rec_arr, thresh_arr = evaluate_full(
    y_test, final_probs_test, 'Meta (Adaptive Thinking)'
)

prec_for_thresh = prec_arr[:-1]
rec_for_thresh = rec_arr[:-1]
mask = prec_for_thresh >= 0.9

## 9. Сравнение с предыдущими версиями

In [ ]:
results = pd.DataFrame([
    {'Модель': 'Baseline CatBoost',  'ROC-AUC': 0.9051, 'PR-AUC': 0.6231, 'Recall@P≥0.9': 0.0145},
    {'Модель': 'SpotFake-Ozon',      'ROC-AUC': 0.9134, 'PR-AUC': 0.6660, 'Recall@P≥0.9': 0.0000},
    {'Модель': 'SAFE-Ozon',       'ROC-AUC': 0.9131, 'PR-AUC': 0.6564, 'Recall@P≥0.9': 0.0012},
    {'Модель': 'Spotfake combined', 'ROC-AUC': 0.9168, 'PR-AUC': 0.6939, 'Recall@P≥0.9': 0.1265},
    {'Модель': 'v4 Mode 1 (Quick Response)',      'ROC-AUC': roc1,   'PR-AUC': pr1,    'Recall@P≥0.9': r1},
    {'Модель': 'v4 Mode 2 (Semantic)',            'ROC-AUC': roc2,   'PR-AUC': pr2,    'Recall@P≥0.9': r2},
    {'Модель': 'v4 Mode 3 (Deep)',                'ROC-AUC': roc3,   'PR-AUC': pr3,    'Recall@P≥0.9': r3},
    {'Модель': 'v4 Meta (Adaptive Thinking)',     'ROC-AUC': roc_f,  'PR-AUC': pr_f,   'Recall@P≥0.9': r_f},
]).set_index('Модель')

results['PR-AUC Δ vs Baseline'] = (results['PR-AUC'] - 0.6231).round(4)

display(results.style
    .highlight_max(subset=['ROC-AUC', 'PR-AUC', 'Recall@P≥0.9'], color='lightgreen')
    .highlight_min(subset=['ROC-AUC', 'PR-AUC'], color='#ffcccc')
    .format('{:.4f}', na_rep='—'))

## 11. Анализ режимов: кому доверяет мета-модель

Проверяем гипотезу из MMD-Thinker: сложные случаи → глубокий режим, простые → быстрый.

In [ ]:
# Для каждого примера: какой режим ближе к финальному предсказанию?
unc_test = 1 - 2 * np.abs(test1 - 0.5)
conf_bins = pd.qcut(unc_test, q=4,
                    labels=['Уверенный Mode1', 'Умеренный', 'Неуверенный', 'Очень неуверенный'])

analysis = pd.DataFrame({
    'uncertainty_bin': conf_bins,
    'Mode1_prob': test1,
    'Mode2_prob': test2,
    'Mode3_prob': test3,
    'Meta_prob':  final_probs_test,
    'delta_M2_M1': np.abs(test2 - test1),
    'delta_M3_M1': np.abs(test3 - test1),
    'true_label': y_test,
})

mode_by_uncertainty = analysis.groupby('uncertainty_bin').agg({
    'delta_M2_M1': 'mean',
    'delta_M3_M1': 'mean',
    'true_label': 'mean',
    'Mode1_prob': 'count',
}).rename(columns={
    'delta_M2_M1': 'Вклад Mode2 поверх Mode1',
    'delta_M3_M1': 'Вклад Mode3 поверх Mode1',
    'true_label': 'Доля контрафакта',
    'Mode1_prob': 'Кол-во примеров',
})

print('Анализ: когда текст/изображение меняют решение относительно Mode 1')
print('(аналог adaptive mode selection из MMD-Thinker)')
display(mode_by_uncertainty.style.format('{:.4f}', na_rep='—'
    ).format({'Кол-во примеров': '{:.0f}'}))

Анализ: когда текст/изображение меняют решение относительно Mode 1
(аналог adaptive mode selection из MMD-Thinker)


,Вклад Mode2 поверх Mode1,Вклад Mode3 поверх Mode1,Доля контрафакта,Кол-во примеров
uncertainty_bin,,,,
Уверенный Mode1,0.001661,0.002062,0.003431,4955
Умеренный,0.008844,0.009194,0.038958,4954
Неуверенный,0.032553,0.037505,0.123940,4954
Очень неуверенный,0.100709,0.117082,0.361857,4955


## 12. Ablation study

In [ ]:
avg_probs = (test1 + test2 + test3) / 3
roc_avg = roc_auc_score(y_test, avg_probs)
pr_avg  = average_precision_score(y_test, avg_probs)
p_avg, r_avg, _ = precision_recall_curve(y_test, avg_probs)
r90_avg = r_avg[p_avg >= 0.9].max() if (p_avg >= 0.9).any() else 0.0
meta_no_sw = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED, class_weight='balanced')
meta_no_sw.fit(meta_train_sc, y_train)
probs_no_sw = meta_no_sw.predict_proba(meta_test_sc)[:, 1]
roc_nsw = roc_auc_score(y_test, probs_no_sw)
pr_nsw  = average_precision_score(y_test, probs_no_sw)
p_nsw, r_nsw, _ = precision_recall_curve(y_test, probs_no_sw)
r90_nsw = r_nsw[p_nsw >= 0.9].max() if (p_nsw >= 0.9).any() else 0.0
ablation = pd.DataFrame([
    {'Конфигурация': 'Только Mode 1 (Quick)',           'ROC-AUC': roc1,   'PR-AUC': pr1,    'Recall@P≥0.9': r1},
    {'Конфигурация': 'Только Mode 2 (+текст)',           'ROC-AUC': roc2,   'PR-AUC': pr2,    'Recall@P≥0.9': r2},
    {'Конфигурация': 'Только Mode 3 (+CLIP)',            'ROC-AUC': roc3,   'PR-AUC': pr3,    'Recall@P≥0.9': r3},
    {'Конфигурация': 'Простой avg 1+2+3',               'ROC-AUC': roc_avg,'PR-AUC': pr_avg, 'Recall@P≥0.9': r90_avg},
    {'Конфигурация': 'Meta без sample weights',         'ROC-AUC': roc_nsw,'PR-AUC': pr_nsw, 'Recall@P≥0.9': r90_nsw},
    {'Конфигурация': 'Meta + sample weights (финал)',   'ROC-AUC': roc_f,  'PR-AUC': pr_f,   'Recall@P≥0.9': r_f},
]).set_index('Конфигурация')

display(ablation.style
    .highlight_max(subset=['ROC-AUC', 'PR-AUC', 'Recall@P≥0.9'], color='lightgreen')
    .format('{:.4f}'))

,ROC-AUC,PR-AUC,Recall@P≥0.9
Конфигурация,,,
Только Mode 1 (Quick),0.9297,0.7131,0.0111
Только Mode 2 (+текст),0.9467,0.7472,0.0814
Только Mode 3 (+CLIP),0.9484,0.7579,0.1181
Простой avg 1+2+3,0.9452,0.7478,0.0741
Meta без sample weights,0.9438,0.7297,0.0000
Meta + sample weights (финал),0.9437,0.7344,0.0004
